# 01 — Exploration

Pair the MSYS2 and MINGW package repositories and look at what the two
ecosystems share.

The inputs are `ls -la` snapshots of the two upstream repositories
(`data/msys2-pkgs.tsv`, `data/mingw-pkgs.tsv`). Only directory entries are
packages; `total`, `LICENSE` and `README.md` lines are metadata and are
ignored by the parser.

In [ ]:
import sys
from pathlib import Path

# Keep the src-layout importable when running from the repository root.
sys.path.insert(0, str(Path.cwd() / "src"))

from msys2_dataset.packages import (
    pair_packages,
    parse_ls_listing_file,
    summarise,
)

In [ ]:
DATA = Path("data")
msys2 = parse_ls_listing_file(DATA / "msys2-pkgs.tsv")
mingw = parse_ls_listing_file(DATA / "mingw-pkgs.tsv")
len(msys2), len(mingw)

### Sanity check the parser

Directories only, no `total` line, no plain files.

In [ ]:
assert not {"total", "LICENSE", "README.md"} & {e.name for e in msys2}
assert not {"total", "LICENSE", "README.md"} & {e.name for e in mingw}
[e.name for e in msys2[:5]]

### Pair by normalised name

`mingw-w64-zlib` and `mingw-w64-ucrt-x86_64-zlib` both normalise to `zlib`.

In [ ]:
pairs = pair_packages(msys2, mingw)
stats = summarise(pairs)
stats

In [ ]:
paired = [p for p in pairs if p.paired]
[f"{p.msys2.name} <-> {p.mingw.name}" for p in paired[:15]]

### Only on one side

In [ ]:
msys2_only = [p.name for p in pairs if p.msys2 and not p.mingw]
mingw_only = [p.name for p in pairs if p.mingw and not p.msys2]
print(f"msys2 only ({len(msys2_only)}):", msys2_only[:10])
print(f"mingw only ({len(mingw_only)}):", mingw_only[:10])

### Overlap ratio

How much of the MSYS2 package set has a MINGW counterpart?

In [ ]:
summary = {
    "coverage of msys2": round(stats["paired"] / (stats["paired"] + stats["msys2_only"]), 3),
    "coverage of mingw": round(stats["paired"] / (stats["paired"] + stats["mingw_only"]), 3),
}
summary